In [2]:
't'

't'

In [8]:
#!/usr/bin/env python
# coding: utf-8

import os
import time
import ee
import geemap
import geopandas as gpd
import pandas as pd

# -------------------- 0) EE AUTH (service account, high-volume) --------------------
SERVICE_ACCOUNT = 'service_account'
SA_KEY_PATH     = 'path_to_json'

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = SA_KEY_PATH
print("Initializing Earth Engine via service account + high-volume endpoint...")
credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, SA_KEY_PATH)
ee.Initialize(credentials)
ee.Initialize(credentials, opt_url='https://earthengine-highvolume.googleapis.com')
print("EE initialized.\n")

# -------------------- 1) I/O PATHS --------------------
input_training_data_path  = '/explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_data_v5.csv'
output_embeddings_path    = '/explore/nobackup/people/spotter5/anna_v/v2/satellite_embeddings_1km.csv'

print(f"Loading source data from: {input_training_data_path}")
df_source = pd.read_csv(
    input_training_data_path,
    usecols=['site_reference', 'latitude', 'longitude', 'year']
)

# Only keep 2017+
df_source = df_source[df_source['year'] >= 2017].copy()

# Clean types
df_source = df_source.dropna(subset=['site_reference', 'latitude', 'longitude', 'year'])
df_source['latitude']  = df_source['latitude'].astype(float)
df_source['longitude'] = df_source['longitude'].astype(float)
df_source['year']      = df_source['year'].astype(int)

# Deduplicate site-year combos
df_source = df_source.drop_duplicates(
    subset=['site_reference', 'year', 'latitude', 'longitude']
).reset_index(drop=True)

print(f"Unique site-year rows (>=2017): {len(df_source)}\n")

# Years present in the CSV
years = sorted(int(y) for y in df_source['year'].unique())
print(f"Years to process (from CSV): {years}\n")

# -------------------- 2) Convert to GeoDataFrame → FeatureCollection --------------------
print("Building GeoDataFrame and FeatureCollection...")

gdf = gpd.GeoDataFrame(
    df_source,
    geometry=gpd.points_from_xy(df_source['longitude'], df_source['latitude']),
    crs="EPSG:4326"
)

# This preserves attributes (site_reference, year, etc.) as properties
fc_all = geemap.geopandas_to_ee(gdf)

print("FeatureCollection created.\n")

# -------------------- 3) EMBEDDING COLLECTION --------------------
COL_ID = 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
se_collection = ee.ImageCollection(COL_ID)

# -------------------- 4) Sample per year using sampleRegions --------------------
all_dfs = []

for yr in years:
    yr = int(yr)
    print(f"Processing year: {yr}")

    # All points for this year
    pts_yr = fc_all.filter(ee.Filter.eq('year', yr))
    num_pts = pts_yr.size().getInfo()
    print(f"  - Points for {yr}: {num_pts}")
    if num_pts == 0:
        print(f"  - WARNING: No points found in FeatureCollection for year {yr}. Skipping.")
        continue

    # Annual images for this year: filter by date range and mosaic
    start = f"{yr}-01-01"
    end   = f"{yr+1}-01-01"
    year_coll = se_collection.filterDate(start, end)

    # If collection is empty, skip
    coll_size = year_coll.size().getInfo()
    if coll_size == 0:
        print(f"  - WARNING: No images in collection for year {yr}. Skipping.")
        continue

    # Mosaic all tiles for that year into a single global image
    img = year_coll.mosaic()

    try:
        band_names = ee.Image(img).bandNames().getInfo()
        if not band_names or len(band_names) < 64:
            print(
                f"  - WARNING: Image for {yr} missing bands "
                f"(found {len(band_names) if band_names else 0}). Skipping."
            )
            continue

        # Sample all points for this year at 1 km scale
        sampled_fc = img.sampleRegions(
            collection = pts_yr,
            scale      = 1000,   # 1 km
            geometries = False,
            tileScale  = 2
        )

        # Check size before doing full getInfo()
        samp_size = sampled_fc.size().getInfo()
        print(f"  - Sampled feature count (reported by EE) for {yr}: {samp_size}")
        if samp_size == 0:
            print(f"  - WARNING: No sampled features returned for year {yr}.")
            continue

        sampled_dict = sampled_fc.getInfo()
        feats = sampled_dict.get('features', [])

        if not feats:
            print(f"  - WARNING: getInfo() returned empty features for {yr}.")
            continue

        rows = [f['properties'] for f in feats]
        df_year = pd.DataFrame(rows)

        # Rename A00..A63 → SE_0..SE_63 to match your previous naming
        rename_map = {f"A{i:02d}": f"SE_{i}" for i in range(64)}
        df_year.rename(columns=rename_map, inplace=True)

        # Keep only relevant columns
        keep_cols = ['site_reference', 'year'] + [f"SE_{i}" for i in range(64)]
        df_year = df_year[[c for c in keep_cols if c in df_year.columns]]

        all_dfs.append(df_year)
        print(f"  - Final rows for {yr}: {len(df_year)}")

    except Exception as e:
        print(f"  - ERROR for year {yr}: {e}")
        time.sleep(3)

print("\nSampling complete.")

# -------------------- 5) Save --------------------
if all_dfs:
    df_out = pd.concat(all_dfs, ignore_index=True)
    df_out.to_csv(output_embeddings_path, index=False)
    print(f"Saved 1 km embeddings to: {output_embeddings_path}")
    print(df_out.head())
else:
    print("No results to save.")


Initializing Earth Engine via service account + high-volume endpoint...
EE initialized.

Loading source data from: /explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_data_v5.csv
Unique site-year rows (>=2017): 17137

Years to process (from CSV): [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Building GeoDataFrame and FeatureCollection...
FeatureCollection created.

Processing year: 2017
  - Points for 2017: 2142
  - Sampled feature count (reported by EE) for 2017: 2140
  - Final rows for 2017: 2140
Processing year: 2018
  - Points for 2018: 2142
  - Sampled feature count (reported by EE) for 2018: 2140
  - Final rows for 2018: 2140
Processing year: 2019
  - Points for 2019: 2142
  - Sampled feature count (reported by EE) for 2019: 2140
  - Final rows for 2019: 2140
Processing year: 2020
  - Points for 2020: 2142
  - Sampled feature count (reported by EE) for 2020: 2140
  - Final rows for 2020: 2140
Processing year: 2021
  - Points for 2021: 2142
  - Sampled feature 

In [5]:
input_training_data_path  = '/explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_data_v5.csv'
output_embeddings_path    = '/explore/nobackup/people/spotter5/anna_v/v2/satellite_embeddings_1km.csv'

print(f"Loading source data from: {input_training_data_path}")
df_source = pd.read_csv(
    input_training_data_path,
    usecols=['site_reference', 'latitude', 'longitude', 'year']
)

# Only keep 2017+
df_source = df_source[df_source['year'] >= 2017].copy()

df_source['year'].unique()

Loading source data from: /explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_data_v5.csv


array([2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025])

In [6]:
import geemap